# Five-move blunder prediction

Production notebook for predicting whether the player who just moved will blunder within their next five own moves.

## Production summary

This notebook trains the five-move blunder-prediction model, tunes the best boosted-tree candidate using validation average precision, calibrates a decision threshold on a separate calibration split, evaluates once on the test split, and exports figures, fitted models, metadata, and the CSV data used to generate each production plot.

In [ ]:
# imports
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd

try:
  from tqdm.auto import tqdm
except ImportError:
  def tqdm(iterable, **kwargs):
    return iterable

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
  ConfusionMatrixDisplay,
  PrecisionRecallDisplay,
  RocCurveDisplay,
  average_precision_score,
  balanced_accuracy_score,
  classification_report,
  confusion_matrix,
  f1_score,
  matthews_corrcoef,
  precision_recall_curve,
  precision_score,
  recall_score,
  roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

# Reproducible presentation defaults.
plt.rcParams.update({
  "figure.dpi": 120,
  "savefig.dpi": 300,
  "axes.spines.top": False,
  "axes.spines.right": False,
  "axes.titleweight": "bold",
  "axes.labelsize": 11,
  "axes.titlesize": 12,
  "legend.frameon": False,
  "font.size": 10,
})

RUN_NAME = "blunder-5-moves"

MAX_ABS_EVAL_PAWNS = 50.0
MISTAKE_PAWN_LOSS = 1.0
BLUNDER_PAWN_LOSS = 2.0
HORIZON_OWN_MOVES = 5
RANDOM_STATE = 42


## 1. Import data

In [ ]:
def find_project_root(start=None):
  if start is None:
    start = Path.cwd()

  start = Path(start).resolve()

  for path in [start, *start.parents]:
    has_pyproject = (path / "pyproject.toml").exists()
    has_data = (path / "data").exists()

    if has_pyproject and has_data:
      return path

  raise FileNotFoundError(
    "Could not find the project root. Run this notebook from "
    "the project or one of its subdirectories."
  )


ROOT = find_project_root()
DATA_DIR = (
  ROOT
  / "data"
  / "processed"
  / "lichess-2017-05-eval-all"
)

GAMES_PATH = DATA_DIR / "games.parquet"
PLIES_PATH = DATA_DIR / "plies.parquet"
FEATURES_PATH = DATA_DIR / "features.parquet"
DICT_PATH = DATA_DIR / "feature_dictionary.csv"

for path in [
  GAMES_PATH,
  PLIES_PATH,
  FEATURES_PATH,
  DICT_PATH,
]:
  print(f"{path.name:24s} exists={path.exists()}")

FIGURES_DIR = ROOT / "figures" / RUN_NAME
MODELS_DIR = ROOT / "models" / RUN_NAME
PLOT_DATA_DIR = ROOT / "plot-data" / RUN_NAME

for directory in [FIGURES_DIR, MODELS_DIR, PLOT_DATA_DIR]:
  directory.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem):
  png_path = FIGURES_DIR / f"{stem}.png"
  pdf_path = FIGURES_DIR / f"{stem}.pdf"
  fig.savefig(png_path, bbox_inches="tight", facecolor="white")
  fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
  print(f"saved: {png_path}")
  print(f"saved: {pdf_path}")


def save_plot_data(df, stem):
  path = PLOT_DATA_DIR / f"{stem}.csv"
  df.to_csv(path, index=False)
  print(f"saved: {path}")
  return path


In [ ]:
# inspect
def require_processed_files(paths, build_hint):
  missing = [path for path in paths if not path.exists()]
  if missing:
    missing_text = "\n".join(f"  - {path}" for path in missing)
    raise FileNotFoundError(
      "Missing processed data files:\n"
      f"{missing_text}\n\n"
      "These parquet files are generated artifacts and are not downloaded automatically.\n"
      f"Regenerate them with:\n{build_hint}"
    )


require_processed_files(
  [GAMES_PATH, PLIES_PATH, FEATURES_PATH],
  """cd blunder
uv run python process_data.py \
  --input data/raw/lichess_db_standard_rated_2017-05.pgn.zst \
  --output-dir data/processed/lichess-2017-05-eval-all \
  --batch-size 1000 \
  --format parquet""",
)

games = pd.read_parquet(GAMES_PATH)
plies = pd.read_parquet(PLIES_PATH)
features = pd.read_parquet(FEATURES_PATH)

if DICT_PATH.exists():
  feature_dict = pd.read_csv(DICT_PATH)
else:
  feature_dict = pd.DataFrame()

print("games:", games.shape)
print("plies:", plies.shape)
print("features:", features.shape)
print("feature_dict:", feature_dict.shape)


## 2. Detect the essential columns

This notebook starts from the dataframes already loaded above. The goal here is
not to create modelling features yet; it is to bind the raw table columns to the
semantic roles the rest of the blunder-prediction pipeline needs.

Ply parity is treated as authoritative for the mover:

- odd ply: White just moved;
- even ply: Black just moved.

An existing turn/side/color column is still detected, but only as a diagnostic,
because such columns may mean either "player who moved" or "player to move next"
depending on the export.


In [ ]:
REQUIRED_DATAFRAMES = ["games", "plies"]


def missing_dataframe(name):
  if name not in globals():
    return True
  return not isinstance(globals()[name], pd.DataFrame)


missing_dataframes = [
  name
  for name in REQUIRED_DATAFRAMES
  if missing_dataframe(name)
]

if missing_dataframes:
  raise NameError(
    "Run the import-data cells first. Missing dataframe(s): "
    + ", ".join(missing_dataframes)
  )

if "features" not in globals() or not isinstance(features, pd.DataFrame):
  features = pd.DataFrame()

# Manual escape hatch. Fill this only if auto-detection picks the wrong column.
# Keys are semantic roles used below; values must be actual columns in that table.
COLUMN_OVERRIDES = {
  "plies": {
    # "game_id": "game_index",
    # "ply": "ply",
    # "eval_pawns": "eval_pawns",
    # "turn": "side",
  },
  "games": {
    # "game_id": "game_index",
  },
  "features": {
    # "game_id": "game_index",
  },
}


def pick_column(
  df,
  candidates,
  *,
  table_name,
  role,
  required=True,
):
  override = COLUMN_OVERRIDES.get(table_name, {}).get(role)

  if override is not None:
    if override in df.columns:
      return override

    raise KeyError(
      f"COLUMN_OVERRIDES[{table_name!r}][{role!r}]={override!r} "
      f"is not present in {table_name}.columns"
    )

  for name in candidates:
    if name in df.columns:
      return name

  if required:
    tried = ", ".join(candidates)
    raise KeyError(
      f"Could not find {role!r} in {table_name}. Tried: {tried}"
    )

  return None


def pick_contains(
  df,
  include,
  *,
  table_name,
  role,
  exclude=None,
  required=True,
):
  if exclude is None:
    exclude = []

  include = [x.lower() for x in include]
  exclude = [x.lower() for x in exclude]

  for col in df.columns:
    low = col.lower()
    has_all = all(x in low for x in include)
    has_excluded = any(x in low for x in exclude)

    if has_all and not has_excluded:
      return col

  if required:
    raise KeyError(
      f"Could not find {role!r} in {table_name} containing {include}"
    )

  return None


GAME_ID_CANDIDATES = ["game_id", "game_index", "game_idx", "id"]
PLY_CANDIDATES = ["ply", "ply_index", "ply_number", "move_ply"]
EVAL_CANDIDATES = [
  "eval_pawns",
  "eval",
  "score",
  "engine_eval",
  "stockfish_eval",
  "stockfish_eval_pawns",
]
TURN_CANDIDATES = ["turn", "side", "color", "player_color", "mover"]

plies_game_id_col = pick_column(
  plies,
  GAME_ID_CANDIDATES,
  table_name="plies",
  role="game_id",
)

ply_col = pick_column(
  plies,
  PLY_CANDIDATES,
  table_name="plies",
  role="ply",
)

eval_col = pick_column(
  plies,
  EVAL_CANDIDATES,
  table_name="plies",
  role="eval_pawns",
  required=False,
)

if eval_col is None:
  eval_col = pick_contains(
    plies,
    ["eval"],
    table_name="plies",
    role="eval_pawns",
    exclude=["mate", "raw", "comment"],
  )

turn_col = pick_column(
  plies,
  TURN_CANDIDATES,
  table_name="plies",
  role="turn",
  required=False,
)

games_game_id_col = pick_column(
  games,
  GAME_ID_CANDIDATES,
  table_name="games",
  role="game_id",
  required=False,
)

features_game_id_col = None
if not features.empty:
  features_game_id_col = pick_column(
    features,
    GAME_ID_CANDIDATES,
    table_name="features",
    role="game_id",
    required=False,
  )

# Backwards-compatible aliases for code adapted from earlier notebooks.
game_id_col = plies_game_id_col

print("plies_game_id_col:   ", plies_game_id_col)
print("ply_col:             ", ply_col)
print("eval_col:            ", eval_col)
print("turn_col diagnostic: ", turn_col)
print("games_game_id_col:   ", games_game_id_col)
print("features_game_id_col:", features_game_id_col)


## 3. Sample complete games before feature construction

For development, sample by game id before constructing move-level targets and
features. This keeps every retained game complete, which matters because the
future 5-move label depends on later plies from the same game.

Set `N_GAMES_SAMPLE = None` to use all games.


In [ ]:
RANDOM_STATE = 42
N_GAMES_SAMPLE = 100_000
RANDOM_GAME_SAMPLE = True


def sample_complete_games(
  plies_df,
  *,
  n_games,
  random_sample=True,
  seed=RANDOM_STATE,
):
  if n_games is None:
    sampled_plies = plies_df.copy()
  else:
    game_ids = plies_df[plies_game_id_col].drop_duplicates()

    if n_games >= len(game_ids):
      selected_game_ids = game_ids
    elif random_sample:
      selected_game_ids = game_ids.sample(
        n=n_games,
        random_state=seed,
      )
    else:
      selected_game_ids = game_ids.head(n_games)

    sampled_plies = plies_df[
      plies_df[plies_game_id_col].isin(selected_game_ids)
    ].copy()

  return sampled_plies.sort_values(
    [plies_game_id_col, ply_col]
  ).reset_index(drop=True)


plies = sample_complete_games(
  plies,
  n_games=N_GAMES_SAMPLE,
  random_sample=RANDOM_GAME_SAMPLE,
)

sampled_game_ids = set(plies[plies_game_id_col].unique())

if games_game_id_col is not None:
  games = games[
    games[games_game_id_col].isin(sampled_game_ids)
  ].copy()

if features_game_id_col is not None:
  features = features[
    features[features_game_id_col].isin(sampled_game_ids)
  ].copy()

print("sampled games:", plies[plies_game_id_col].nunique())
print("sampled plies:", len(plies))
print("games rows:", len(games))
print("features rows:", len(features))


## 4. Construct move loss column
Be careful about alignment here.
* `evaluation before move = pervious row's evaluation`
* `evaluation after move = current row's evaluation`

In [ ]:
def add_move_columns(df):

  # Prepare the dataframe so that later calculations
  # can happen in the correct move order without modifying
  # the original input. (safety feature)
  df = df.copy()
  df = df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)

  # Extracting relevant features into new dataframe.
  df["ply_number"] = pd.to_numeric(
    df[ply_col],
    errors="coerce"  # NaN for errors
  )

  df["eval_pawns"] = pd.to_numeric(
    df[eval_col],
    errors="coerce",
  ).clip(
    -MAX_ABS_EVAL_PAWNS,
    MAX_ABS_EVAL_PAWNS
  )

  df["mover"] = np.where(
    df["ply_number"] % 2 == 1,
    "white",
    "black",
  )

  df["fullmove_number"] = (
    (df["ply_number"] + 1) // 2
  ).astype("Int64")

  group = df.groupby(game_id_col, sort=False)
  df["eval_before_pawns"] = group["eval_pawns"].shift(1)
  df["eval_delta_pawns"] = (
    df["eval_pawns"] - df["eval_before_pawns"]
  )

  white_loss = -df["eval_delta_pawns"]
  black_loss = df["eval_delta_pawns"]

  df["pawn_loss"] = np.where(
    df["mover"].eq("white"),
    white_loss,
    black_loss,
  )
  df["pawn_loss"] = df["pawn_loss"].clip(lower=0)

  df["is_mistake"] = df["pawn_loss"].ge(
    MISTAKE_PAWN_LOSS
  ).astype("Int64")

  df["is_blunder"] = df["pawn_loss"].ge(
    BLUNDER_PAWN_LOSS
  ).astype("Int64")

  missing_pair = (
    df["eval_before_pawns"].isna()
    | df["eval_pawns"].isna()
  )
  df.loc[missing_pair, "pawn_loss"] = np.nan
  df.loc[missing_pair, "is_mistake"] = pd.NA
  df.loc[missing_pair, "is_blunder"] = pd.NA

  return df


plies_ml = add_move_columns(plies)


In [ ]:
# Safety function supplied by the chat to deal with
# missing/corrupted values.
def normalize_turn_value(value):
  if pd.isna(value):
    return np.nan

  value = str(value).strip().lower()

  if value in {"w", "white", "1", "true"}:
    return "white"

  if value in {"b", "black", "0", "false"}:
    return "black"

  return value


if turn_col is not None:
  supplied_turn = plies_ml[turn_col].map(normalize_turn_value)
  direct_match = supplied_turn.eq(plies_ml["mover"]).mean()
  opposite_match = supplied_turn.ne(plies_ml["mover"]).mean()

  print("Turn-column diagnostic")
  print("matches mover from parity: ", direct_match)
  print("differs from parity mover:  ", opposite_match)

show_cols = [
  game_id_col,
  ply_col,
  "mover",
  "eval_before_pawns",
  "eval_pawns",
  "eval_delta_pawns",
  "pawn_loss",
  "is_blunder",
]

display(plies_ml[show_cols].head(100))


In [ ]:
# Sanity check: what fraction of moves are mistakes/blunders
print("Move-level event rates")
display(
  plies_ml[["is_mistake", "is_blunder"]]
  .astype(float)
  .mean()
  .to_frame("fraction")
)

# And also the largest measured pawn losses
# remember we clip at MAX_ABS_EVAL_PAWNS
print("Largest measured pawn losses")
display(
  plies_ml[show_cols]
  .sort_values("pawn_loss", ascending=False)
  .head(20)
)


## 5. Build targets

The prediction question is evaluated after a player has just made a move:

> Will this same player make at least one engine-defined blunder during their
> next five own moves?

The current move is not part of the target. For each row, the notebook looks
forward only within the same `(game_index, mover)` group, so White rows look at
White's next moves and Black rows look at Black's next moves.

`future_blunder_count` is the number of blunders in those next five own moves.
For example, if the next five own moves have blunder labels `[0, 1, 0, 1, 0]`,
then `future_blunder_count == 2` and `will_blunder_soon == 1`.

`will_blunder_soon` is the binary modelling target:

- `1`: at least one of the next five own moves is a blunder;
- `0`: all five next own moves are observed and none is a blunder;
- `NaN`: the target is unknown/censored.

A missing target is expected near the end of games, because the player may not
have five later moves. It can also happen when one of the future moves exists
but has no usable blunder label, usually because the evaluation needed to
compute pawn loss is missing. Those rows are not safe negatives; they are rows
where the target is not observed.


In [ ]:
def add_future_target(df, horizon):
  df = df.copy()

  # Put each player's moves in chronological order within each game. This makes
  # group.shift(-1) mean "this same player's next move", not simply the next ply.
  df = df.sort_values(
    [game_id_col, "mover", ply_col]
  )

  keys = [game_id_col, "mover"]
  group = df.groupby(keys, sort=False)["is_blunder"]

  # future_1 is the same player's next move, future_2 is their move after that,
  # and so on up to the five-own-move prediction horizon.
  future_labels = []
  for step in range(1, horizon + 1):
    future_labels.append(group.shift(-step).rename(f"future_{step}"))

  future = pd.concat(future_labels, axis=1)

  # A row has an observed target only if all five future blunder labels are
  # present. If the game ends first, or if a future eval/blunder label is
  # missing, the target is unknown rather than negative.
  complete_window = future.notna().all(axis=1)

  # Count blunders in the next five own moves. min_count=horizon keeps this as
  # NaN whenever the full five-move window is not observed.
  df["future_blunder_count"] = future.sum(
    axis=1,
    min_count=horizon,
  )

  # Binary version of the count used as the model target.
  df["will_blunder_soon"] = np.where(
    complete_window,
    future.gt(0).any(axis=1).astype(float),
    np.nan,
  )

  return df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)


plies_ml = add_future_target(
  plies_ml,
  HORIZON_OWN_MOVES,
)

print("Target counts, including unknown/censored rows")
display(
  plies_ml["will_blunder_soon"]
  .value_counts(dropna=False)
)

# Explain why targets are missing. This is diagnostic only; these quantities use
# future information and must not be used as model features.
target_ordered = plies_ml.sort_values(
  [game_id_col, "mover", ply_col]
).copy()
target_group = target_ordered.groupby(
  [game_id_col, "mover"],
  sort=False,
)["is_blunder"]
future_for_diagnostics = pd.concat(
  [
    target_group.shift(-step).rename(f"future_{step}")
    for step in range(1, HORIZON_OWN_MOVES + 1)
  ],
  axis=1,
)
later_own_moves_available = target_ordered.groupby(
  [game_id_col, "mover"],
  sort=False,
).cumcount(ascending=False)

missing_target = target_ordered["will_blunder_soon"].isna()
end_of_game_censored = missing_target & (
  later_own_moves_available < HORIZON_OWN_MOVES
)
missing_future_label = missing_target & (
  later_own_moves_available >= HORIZON_OWN_MOVES
)

print("Why will_blunder_soon is missing")
display(
  pd.Series({
    "fewer_than_5_later_own_moves": int(end_of_game_censored.sum()),
    "5_later_moves_exist_but_a_future_label_is_missing": int(
      missing_future_label.sum()
    ),
    "missing_targets_total": int(missing_target.sum()),
  }).to_frame("rows")
)

print("Missing future blunder labels by step among missing targets")
display(
  future_for_diagnostics[missing_target]
  .isna()
  .sum()
  .to_frame("missing_labels")
)


### Concrete target-window example

The current row's own `is_blunder` value is not enough to define the target.
`will_blunder_soon` depends on the next five rows for the same `(game_index, mover)`.

For the first sampled game, rows around index 20 look surprising because the
current rows have valid `pawn_loss` and `is_blunder` values. The target is still
missing because at least one move inside the future five-own-move window has a
missing `is_blunder` label, or because fewer than five later own moves exist.


In [ ]:
target_example_cols = [
  game_id_col,
  ply_col,
  "mover",
  "pawn_loss",
  "is_blunder",
  "future_blunder_count",
  "will_blunder_soon",
]

print("Rows 20-29 from plies_ml")
display(plies_ml.loc[20:29, target_example_cols])


def show_future_window(row_index):
  row = plies_ml.loc[row_index]
  future_window = (
    plies_ml[
      plies_ml[game_id_col].eq(row[game_id_col])
      & plies_ml["mover"].eq(row["mover"])
      & plies_ml[ply_col].gt(row[ply_col])
    ]
    .sort_values(ply_col)
    .head(HORIZON_OWN_MOVES)
  )

  print(
    f"row {row_index}: game={row[game_id_col]}, ply={row[ply_col]}, "
    f"mover={row['mover']}"
  )
  print(
    f"future rows found: {len(future_window)} / {HORIZON_OWN_MOVES}; "
    f"target={row['will_blunder_soon']}; "
    f"future_blunder_count={row['future_blunder_count']}"
  )
  display(future_window[target_example_cols])


show_future_window(20)
show_future_window(24)


One final comment here: I expect a lot of the missing evals to come from an artifact of lichess's
eval system. Usually lichess gives `%eval` with a score, but when the engine expects a mate in 
`n` moves, it will instead give `%eval #n`, which leads to missing evals. 

## 6. Construct strictly causal features

All features below should be known immediately after the current move. That
means the model may use the current board/eval/clock state and the player's
history up to this move, but not anything that happens later in the game.

The earlier version used a coarse categorical `phase` based on move number. That
is a weak proxy: move 20 can be a queenless endgame or a fully loaded middlegame.
The processed `plies` table already has board-derived phase signals, so use
those instead:

- `phase_progress`: fraction of starting non-pawn material that has disappeared;
- `opening_like_weight`, `middlegame_like_weight`, `endgame_like_weight`: smooth
  phase weights derived from the current board state.

Do not use the phase summaries in the game-level `features` table here, such as
`final_phase_progress` or `mean_phase_progress`. Those summarize the whole game
and would leak future information into a move-level prediction task.


In [ ]:
def add_causal_features(df):
  df = df.copy()
  df = df.sort_values(
    [game_id_col, ply_col]
  ).reset_index(drop=True)

  game_group = df.groupby(game_id_col, sort=False)
  player_keys = [game_id_col, "mover"]
  player_group = df.groupby(player_keys, sort=False)

  mover_sign = np.where(df["mover"].eq("white"), 1.0, -1.0)
  df["mover_is_white"] = df["mover"].eq("white").astype(int)

  # Position/eval immediately after the current move, from the mover's
  # perspective. Positive means the mover stands better.
  df["eval_for_mover"] = df["eval_pawns"] * mover_sign
  df["eval_before_for_mover"] = df["eval_before_pawns"] * mover_sign
  df["abs_eval_pawns"] = df["eval_pawns"].abs()

  # Current move quality. This is causal because the prediction is made after
  # the move has been played and evaluated.
  df["current_pawn_loss"] = df["pawn_loss"]
  df["current_is_mistake"] = df["is_mistake"].astype(float)
  df["current_is_blunder"] = df["is_blunder"].astype(float)
  df["current_eval_swing"] = df["eval_delta_pawns"]
  df["current_eval_swing_for_mover"] = df["eval_delta_pawns"] * mover_sign
  df["abs_current_eval_swing"] = df["current_eval_swing"].abs()

  # Board-derived phase and material signals already exist in the plies table.
  # They describe the current board, unlike full-game summaries in features.
  df["phase_progress_current"] = df["phase_progress"]
  df["opening_weight_current"] = df["opening_like_weight"]
  df["middlegame_weight_current"] = df["middlegame_like_weight"]
  df["endgame_weight_current"] = df["endgame_like_weight"]
  df["material_imbalance_for_mover"] = (
    df["material_imbalance_white"] * mover_sign
  )
  df["non_pawn_imbalance_for_mover"] = (
    df["non_pawn_imbalance_white"] * mover_sign
  )
  df["total_material_current"] = df["total_material"]
  df["total_non_pawn_material_current"] = df["total_non_pawn_material"]
  df["both_queens_present_current"] = (
    df["both_queens_present"].astype(float)
  )
  df["no_queens_present_current"] = df["no_queens_present"].astype(float)

  # Clock pressure after the current move. Low clock is an obvious candidate for
  # future blunders and is available at prediction time.
  df["clock_seconds_current"] = df["clock_seconds"]
  df["log_clock_seconds_current"] = np.log1p(df["clock_seconds"])
  df["previous_own_clock_seconds"] = player_group[
    "clock_seconds"
  ].shift(1)
  df["own_clock_change"] = (
    df["clock_seconds"] - df["previous_own_clock_seconds"]
  )

  # Historical own-move quality. Shift first so these summaries never include
  # the current row unless explicitly named current_* above.
  previous_loss = player_group["pawn_loss"].shift(1)
  previous_mistake = player_group["is_mistake"].shift(1)
  previous_blunder = player_group["is_blunder"].shift(1)

  hist_keys = [df[game_id_col], df["mover"]]
  df["previous_own_pawn_loss"] = previous_loss
  df["previous_own_mistakes"] = (
    previous_mistake.fillna(0).groupby(hist_keys, sort=False).cumsum()
  )
  df["previous_own_blunders"] = (
    previous_blunder.fillna(0).groupby(hist_keys, sort=False).cumsum()
  )

  for window in [3, 5, 10]:
    rolled = (
      previous_loss
      .groupby(hist_keys, sort=False)
      .rolling(window, min_periods=1)
      .agg(["mean", "max", "std"])
      .reset_index(level=[0, 1], drop=True)
    )

    df[f"previous_loss_mean_{window}"] = rolled["mean"]
    df[f"previous_loss_max_{window}"] = rolled["max"]
    df[f"previous_loss_std_{window}"] = rolled["std"]

  # A move-number feature can still be useful as tempo/elapsed-game information,
  # but it is no longer asked to stand in for chess phase.
  df["fullmove_number_log"] = np.log1p(
    df["fullmove_number"].astype(float)
  )

  return df


plies_ml = add_causal_features(plies_ml)


In [ ]:
candidate_features = [
  # Side and elapsed-game context
  "mover_is_white",
  "fullmove_number",
  "fullmove_number_log",

  # Current eval and move quality
  "eval_for_mover",
  "eval_before_for_mover",
  "abs_eval_pawns",
  "current_pawn_loss",
  "current_is_mistake",
  "current_is_blunder",
  "current_eval_swing_for_mover",
  "abs_current_eval_swing",

  # Current board phase/material, replacing the old move-number phase bucket
  "phase_progress_current",
  "opening_weight_current",
  "middlegame_weight_current",
  "endgame_weight_current",
  "material_imbalance_for_mover",
  "non_pawn_imbalance_for_mover",
  "total_material_current",
  "total_non_pawn_material_current",
  "both_queens_present_current",
  "no_queens_present_current",

  # Clock pressure
  "clock_seconds_current",
  "log_clock_seconds_current",
  "previous_own_clock_seconds",
  "own_clock_change",

  # Player's own recent error history before the current move
  "previous_own_pawn_loss",
  "previous_own_mistakes",
  "previous_own_blunders",
  "previous_loss_mean_3",
  "previous_loss_max_3",
  "previous_loss_std_3",
  "previous_loss_mean_5",
  "previous_loss_max_5",
  "previous_loss_std_5",
  "previous_loss_mean_10",
  "previous_loss_max_10",
  "previous_loss_std_10",
]

feature_cols = [
  col for col in candidate_features
  if col in plies_ml.columns
]

print(f"{len(feature_cols)} candidate features")
for col in feature_cols:
  print(col)


## 7. Create the modelling table

This section turns the labelled move table into the objects used by the ML code:

- `model_df`: one row per move with an observed target;
- `X`: the candidate feature matrix;
- `y`: the binary target, `will_blunder_soon`;
- `groups`: the game id for each row, used later so whole games stay together
  in train/validation/test splits.

Rows with `will_blunder_soon = NaN` are removed because their future five-own-move
window is not fully observed. They are not negative examples.

Unlike notebook 09, this version does not drop rows just because a feature is
missing. Some missing feature values are structural and expected: the first move
has no previous eval, early moves have no rolling-history window yet, and mate
sequences may lack numeric evals. Those missing feature values should be handled
inside the fitted preprocessing pipeline, usually with an imputer. Dropping them
here would throw away valid labelled examples and bias the dataset toward longer,
cleaner games.


In [ ]:
model_df = plies_ml[
  plies_ml["will_blunder_soon"].notna()
].copy()

X = model_df[feature_cols]
y = model_df["will_blunder_soon"].astype(int)
groups = model_df[game_id_col]

print("model rows:", len(model_df))
print("unique games:", groups.nunique())
print("natural positive prevalence:", y.mean())

print("Target class counts")
display(y.value_counts().sort_index().to_frame("rows"))

print("Feature missingness, highest first")
feature_missingness = (
  X.isna()
  .mean()
  .sort_values(ascending=False)
  .to_frame("missing_fraction")
)
display(feature_missingness.head(20))

print("Rows retained from labelled target table:")
labelled_rows = plies_ml["will_blunder_soon"].notna().sum()
print(f"{len(model_df):,} / {labelled_rows:,}")


## 8. Split by complete games

The train/validation/test split must happen by game, not by row. Rows from the
same chess game are highly related: neighbouring plies share position, clock,
players, opening, and future game context. If one game appeared in both train
and validation/test, the evaluation would be too optimistic.

The split below first creates a one-row-per-game table and marks whether each
game has at least one positive labelled row. It then stratifies on that game
level flag so all splits get a reasonable share of games containing positives,
while still keeping each game entirely in one split.

Validation and test are left at their natural row-level prevalence. They are not
balanced here; balancing/downsampling, if used, should happen only inside the
training split.


In [ ]:
def game_train_valid_calibration_test_split(
  X,
  y,
  groups,
  seed=RANDOM_STATE,
):
  """Split complete games into 70/10/10/10 row-independent roles."""
  game_table = (
    pd.DataFrame({
      "game_id": groups,
      "has_positive": y,
    })
    .groupby("game_id", as_index=False)["has_positive"]
    .max()
  )

  train_games, remainder_games = train_test_split(
    game_table,
    train_size=0.70,
    random_state=seed,
    stratify=game_table["has_positive"],
  )

  valid_games, cal_test_games = train_test_split(
    remainder_games,
    train_size=1 / 3,
    random_state=seed,
    stratify=remainder_games["has_positive"],
  )

  calibration_games, test_games = train_test_split(
    cal_test_games,
    train_size=0.50,
    random_state=seed,
    stratify=cal_test_games["has_positive"],
  )

  game_ids = {
    "train": set(train_games["game_id"]),
    "valid": set(valid_games["game_id"]),
    "calibration": set(calibration_games["game_id"]),
    "test": set(test_games["game_id"]),
  }

  split_names = list(game_ids)
  for i, left in enumerate(split_names):
    for right in split_names[i + 1:]:
      assert game_ids[left].isdisjoint(game_ids[right])

  output = []
  for split_name in split_names:
    mask = groups.isin(game_ids[split_name])
    output.extend([
      X.loc[mask].copy(),
      y.loc[mask].copy(),
      groups.loc[mask].copy(),
    ])

  return tuple(output)


(
  X_train,
  y_train,
  g_train,
  X_valid,
  y_valid,
  g_valid,
  X_calibration,
  y_calibration,
  g_calibration,
  X_test,
  y_test,
  g_test,
) = game_train_valid_calibration_test_split(X, y, groups)

split_summary = pd.DataFrame([
  {
    "split": name,
    "rows": len(split_y),
    "games": split_g.nunique(),
    "positive_prevalence": split_y.mean(),
  }
  for name, split_y, split_g in [
    ("train", y_train, g_train),
    ("valid", y_valid, g_valid),
    ("calibration", y_calibration, g_calibration),
    ("test", y_test, g_test),
  ]
])

display(split_summary)
print("All complete-game split overlap checks passed.")


## 9. Training-only imbalance strategies

For the current five-own-move target, the training labels are not severely
imbalanced: positives are roughly one third of the labelled rows. Downsampling
would throw away a large amount of useful negative data and would make predicted
probabilities harder to interpret.

Use only full-training-data strategies here:

1. **Unweighted full data:** fit on the natural training distribution.
2. **Weighted full data:** keep every row, but weight classes inversely to their
   training frequency so the objective gives positives and negatives equal total
   weight.

The weighted version is useful preparation for a later one-move target, where
the positive rate may be closer to 10%. Importantly, weighting is computed from
`y_train` only. Validation and test remain unweighted and naturally distributed.


In [ ]:
def make_balanced_sample_weights(y_values):
  n_pos = y_values.eq(1).sum()
  n_neg = y_values.eq(0).sum()

  if n_pos == 0 or n_neg == 0:
    raise ValueError("Both classes are required for weighting.")

  pos_weight = len(y_values) / (2 * n_pos)
  neg_weight = len(y_values) / (2 * n_neg)

  return np.where(
    y_values.to_numpy() == 1,
    pos_weight,
    neg_weight,
  )


sample_weight_train_balanced = make_balanced_sample_weights(y_train)

training_variants = {
  "full_unweighted": {
    "X": X_train,
    "y": y_train,
    "sample_weight": None,
  },
  "full_weighted": {
    "X": X_train,
    "y": y_train,
    "sample_weight": sample_weight_train_balanced,
  },
}

print("training rows:", len(y_train))
print("training positives:", int(y_train.eq(1).sum()))
print("training negatives:", int(y_train.eq(0).sum()))
print("training prevalence:", y_train.mean())

weight_summary = pd.DataFrame({
  "class": [0, 1],
  "rows": [int(y_train.eq(0).sum()), int(y_train.eq(1).sum())],
  "sample_weight": [
    sample_weight_train_balanced[y_train.to_numpy() == 0][0],
    sample_weight_train_balanced[y_train.to_numpy() == 1][0],
  ],
  "total_weight": [
    sample_weight_train_balanced[y_train.to_numpy() == 0].sum(),
    sample_weight_train_balanced[y_train.to_numpy() == 1].sum(),
  ],
})

display(weight_summary)


## 10. Preprocessing and candidate models (BDT: XGBoost, LightGBM)

This section defines how raw dataframe columns are converted into model-ready
inputs, then creates the candidate classifiers.

**Imputing.** Some feature values are missing for legitimate reasons: first moves
lack prior-history features, early moves lack rolling-window standard deviations,
clock values can be absent, and mate-sequence evals can remove numeric evals.
Most sklearn estimators cannot accept `NaN` directly, so the preprocessing step
fills missing numeric values with the training median. The median is learned only
from the training split inside the pipeline, which avoids leaking validation/test
statistics into training.

**One-hot encoding.** Tree libraries need numeric matrices. If we later add a
categorical feature, such as opening family or time-control bucket, one-hot
encoding turns each category into a binary indicator column. `handle_unknown="ignore"`
means validation/test categories that were not seen during training get all-zero
indicator values instead of crashing the pipeline.

For this notebook the main candidates are boosted decision trees: XGBoost and
LightGBM. Each is trained on the full training set in two variants:

- unweighted: natural training distribution;
- weighted: same rows, but class-balanced sample weights from section 9.

The dummy prior model is included only as a sanity-check baseline.

The explicit `to_numpy` step after preprocessing is there to avoid a noisy
LightGBM/sklearn warning about feature names. The model still trains on the same
columns in the same order; we simply make sure LightGBM sees the same unnamed
array format during fit, validation scoring, test scoring, and permutation
importance.

Each model gets its own fresh preprocessing object. Reusing a single fitted
`ColumnTransformer` across several pipelines is subtle and risky because fitting
one pipeline can mutate the preprocessing state seen by another pipeline.


In [ ]:
numeric_cols = [
  col for col in feature_cols
  if pd.api.types.is_numeric_dtype(X_train[col])
]

categorical_cols = [
  col for col in feature_cols
  if col not in numeric_cols
]


def make_preprocess():
  # Numeric imputation replaces missing values with the median value learned from
  # X_train. This keeps rows with valid targets even when some features are missing.
  numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
  ])

  # Categorical imputation fills missing categories, then one-hot encoding expands
  # categories into binary numeric columns. There are currently no categorical
  # features in feature_cols, but this keeps the pipeline ready for later additions.
  categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
      handle_unknown="ignore",
      sparse_output=False,
    )),
  ])

  return ColumnTransformer([
    ("numeric", numeric_preprocess, numeric_cols),
    ("categorical", categorical_preprocess, categorical_cols),
  ])


def dataframe_to_numpy(X):
  # LightGBM's sklearn wrapper warns if it is fitted with feature names and later
  # receives an unnamed array. The ColumnTransformer/OneHotEncoder stack can
  # change whether the intermediate object has pandas column names depending on
  # sklearn output configuration. Coerce the preprocessed matrix to NumPy so fit,
  # predict, and permutation-importance calls all use the same unnamed format.
  if hasattr(X, "to_numpy"):
    return X.to_numpy()

  return X


def make_to_numpy():
  return FunctionTransformer(
    dataframe_to_numpy,
    validate=False,
  )


def make_xgboost_model(model_params=None):
  params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_estimators": 450,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 20,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
  }

  if model_params is not None:
    params.update(model_params)

  return Pipeline([
    ("preprocess", make_preprocess()),
    ("to_numpy", make_to_numpy()),
    ("model", XGBClassifier(**params)),
  ])


def make_lightgbm_model(model_params=None):
  params = {
    "objective": "binary",
    "n_estimators": 600,
    "learning_rate": 0.04,
    "num_leaves": 63,
    "min_child_samples": 80,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
  }

  if model_params is not None:
    params.update(model_params)

  return Pipeline([
    ("preprocess", make_preprocess()),
    ("to_numpy", make_to_numpy()),
    ("model", LGBMClassifier(**params)),
  ])


models = {
  "dummy_prior": {
    "pipeline": Pipeline([
      ("preprocess", make_preprocess()),
      ("to_numpy", make_to_numpy()),
      ("model", DummyClassifier(strategy="prior")),
    ]),
    "training_variant": "full_unweighted",
  },
  "xgboost_unweighted": {
    "pipeline": make_xgboost_model(),
    "training_variant": "full_unweighted",
  },
  "xgboost_weighted": {
    "pipeline": make_xgboost_model(),
    "training_variant": "full_weighted",
  },
  "lightgbm_unweighted": {
    "pipeline": make_lightgbm_model(),
    "training_variant": "full_unweighted",
  },
  "lightgbm_weighted": {
    "pipeline": make_lightgbm_model(),
    "training_variant": "full_weighted",
  },
}

print("numeric features:", len(numeric_cols))
print("categorical features:", len(categorical_cols))
print("candidate models:", list(models))


## 11. Train candidate models on the natural validation distribution

The validation metric for model selection is average precision. Candidate models are trained on the full training split, with weighted variants receiving class-balanced sample weights computed only from the training labels.

In [ ]:
def get_score(model, X_eval):
  if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(X_eval)
    positive_index = list(model.classes_).index(1)
    return probabilities[:, positive_index]

  score = model.decision_function(X_eval)
  return 1 / (1 + np.exp(-score))


def threshold_metric_table(y_true, score):
  y_array = np.asarray(y_true, dtype=np.int8)
  score_array = np.asarray(score, dtype=float)

  if len(y_array) != len(score_array):
    raise ValueError("y_true and score must have the same length.")

  order = np.argsort(score_array)[::-1]
  sorted_score = score_array[order]
  sorted_y = y_array[order]
  distinct_ends = np.r_[
    np.flatnonzero(np.diff(sorted_score)),
    len(sorted_score) - 1,
  ]

  threshold_desc = sorted_score[distinct_ends]
  predicted_positive = distinct_ends + 1
  true_positive = np.cumsum(sorted_y)[distinct_ends].astype(float)
  false_positive = predicted_positive - true_positive
  total_positive = float(sorted_y.sum())
  total_negative = float(len(sorted_y) - total_positive)
  false_negative = total_positive - true_positive
  true_negative = total_negative - false_positive

  precision = np.divide(
    true_positive,
    predicted_positive,
    out=np.zeros_like(true_positive, dtype=float),
    where=predicted_positive != 0,
  )
  recall = np.divide(
    true_positive,
    total_positive,
    out=np.zeros_like(true_positive, dtype=float),
    where=total_positive != 0,
  )
  f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(precision, dtype=float),
    where=(precision + recall) != 0,
  )
  denominator = np.sqrt(
    (true_positive + false_positive)
    * (true_positive + false_negative)
    * (true_negative + false_positive)
    * (true_negative + false_negative)
  )
  mcc = np.divide(
    true_positive * true_negative - false_positive * false_negative,
    denominator,
    out=np.zeros_like(true_positive, dtype=float),
    where=denominator != 0,
  )

  table = pd.DataFrame({
    "threshold": threshold_desc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predicted_positive_rate": predicted_positive / len(sorted_y),
    "mcc": mcc,
  })
  return table.sort_values("threshold", ignore_index=True)


def best_threshold_row(threshold_table, metric):
  if metric == "f1":
    sort_cols = ["f1", "precision", "threshold"]
  elif metric == "mcc":
    sort_cols = ["mcc", "f1", "precision", "threshold"]
  else:
    raise ValueError(f"Unsupported threshold metric: {metric}")

  return threshold_table.sort_values(
    sort_cols,
    ascending=False,
  ).iloc[0]


def ranking_metrics(name, model, X_eval, y_eval):
  score = get_score(model, X_eval)
  prevalence = y_eval.mean()
  average_precision = average_precision_score(y_eval, score)
  return {
    "model": name,
    "roc_auc": roc_auc_score(y_eval, score),
    "average_precision": average_precision,
    "prevalence": prevalence,
    "average_precision_lift_over_random": (
      average_precision / prevalence
    ),
  }


WORKFLOW_CACHE_PATH = MODELS_DIR / "workflow_cache.joblib"
workflow_cache = None
if WORKFLOW_CACHE_PATH.exists():
  workflow_cache = joblib.load(WORKFLOW_CACHE_PATH)
  print(f"loaded: {WORKFLOW_CACHE_PATH}")

if workflow_cache is not None:
  fitted_models = workflow_cache["fitted_models"]
  results_df = workflow_cache["results_df"]
  training_timings_df = workflow_cache.get(
    "training_timings_df",
    pd.DataFrame(),
  )
  optimized_model_name = workflow_cache["optimized_model_name"]
  best_model_name = workflow_cache["best_model_name"]
  if "dummy_prior" not in fitted_models:
    dummy_model = models["dummy_prior"]["pipeline"]
    dummy_model.fit(X_train, y_train)
    fitted_models["dummy_prior"] = dummy_model
  SKIP_TRAINING_AND_OPTIMIZATION = True
  display(training_timings_df)
  display(results_df)
else:
  SKIP_TRAINING_AND_OPTIMIZATION = False
  fitted_models = {}
  results = []
  training_timings = []

  for name, spec in tqdm(list(models.items()), desc="Candidate models"):
    print(f"Training {name}...")
    model = spec["pipeline"]
    variant = training_variants[spec["training_variant"]]
    fit_kwargs = {}
    if variant["sample_weight"] is not None:
      fit_kwargs["model__sample_weight"] = variant["sample_weight"]

    fit_start = perf_counter()
    model.fit(variant["X"], variant["y"], **fit_kwargs)
    fit_seconds = perf_counter() - fit_start

    metric_start = perf_counter()
    model_metrics = ranking_metrics(name, model, X_valid, y_valid)
    metric_seconds = perf_counter() - metric_start

    fitted_models[name] = model
    results.append(model_metrics)
    training_timings.append({
      "model": name,
      "fit_seconds": fit_seconds,
      "metric_seconds": metric_seconds,
      "total_seconds": fit_seconds + metric_seconds,
    })
    print(
      f"Finished {name}: fit {fit_seconds:.1f}s, "
      f"metrics {metric_seconds:.1f}s"
    )

  results_df = pd.DataFrame(results).sort_values(
    "average_precision",
    ascending=False,
  )
  training_timings_df = pd.DataFrame(training_timings)
  save_plot_data(results_df, "validation_baseline_model_metrics")
  save_plot_data(training_timings_df, "baseline_training_timings")
  display(training_timings_df)
  display(results_df)


## 12. Hyperparameter optimization using average precision

Only the best non-dummy boosted-tree candidate is tuned. The objective is validation average precision, which matches the ranking goal for rare future-blunder risk.

In [ ]:
N_OPTUNA_TRIALS = 30
OPTUNA_TRAIN_SAMPLE_ROWS = 500_000
OPTUNA_VALID_SAMPLE_ROWS = 200_000


def choose_base_model_for_hpo(results_table):
  non_dummy = results_table[
    ~results_table["model"].str.startswith("dummy")
  ]
  if non_dummy.empty:
    raise ValueError(
      "No boosted-tree model is available for optimization."
    )
  return non_dummy.iloc[0]["model"]


def family_for_model(model_name):
  if model_name.startswith("lightgbm"):
    return "lightgbm"
  if model_name.startswith("xgboost"):
    return "xgboost"
  raise ValueError(f"Unsupported model family: {model_name}")


def suggest_lightgbm_params(trial):
  return {
    "n_estimators": trial.suggest_int(
      "n_estimators", 300, 1200, step=100,
    ),
    "learning_rate": trial.suggest_float(
      "learning_rate", 0.015, 0.10, log=True,
    ),
    "num_leaves": trial.suggest_int(
      "num_leaves", 31, 255, log=True,
    ),
    "min_child_samples": trial.suggest_int(
      "min_child_samples", 30, 300,
    ),
    "subsample": trial.suggest_float("subsample", 0.70, 1.00),
    "colsample_bytree": trial.suggest_float(
      "colsample_bytree", 0.60, 1.00,
    ),
    "reg_alpha": trial.suggest_float(
      "reg_alpha", 1e-4, 10.0, log=True,
    ),
    "reg_lambda": trial.suggest_float(
      "reg_lambda", 1e-3, 30.0, log=True,
    ),
  }


def suggest_xgboost_params(trial):
  return {
    "n_estimators": trial.suggest_int(
      "n_estimators", 300, 1000, step=100,
    ),
    "learning_rate": trial.suggest_float(
      "learning_rate", 0.015, 0.10, log=True,
    ),
    "max_depth": trial.suggest_int("max_depth", 3, 8),
    "min_child_weight": trial.suggest_float(
      "min_child_weight", 3.0, 80.0, log=True,
    ),
    "subsample": trial.suggest_float("subsample", 0.70, 1.00),
    "colsample_bytree": trial.suggest_float(
      "colsample_bytree", 0.60, 1.00,
    ),
    "reg_alpha": trial.suggest_float(
      "reg_alpha", 1e-4, 10.0, log=True,
    ),
    "reg_lambda": trial.suggest_float(
      "reg_lambda", 1e-3, 30.0, log=True,
    ),
  }


def make_hpo_model(family, params):
  if family == "lightgbm":
    return make_lightgbm_model(params)
  if family == "xgboost":
    return make_xgboost_model(params)
  raise ValueError(f"Unknown model family: {family}")


if SKIP_TRAINING_AND_OPTIMIZATION:
  print("Skipping hyperparameter optimization; cache is loaded.")
  best_params_df = pd.DataFrame()
  hpo_trials_df = pd.DataFrame()
else:
  hpo_base_model_name = choose_base_model_for_hpo(results_df)
  hpo_training_variant_name = models[hpo_base_model_name][
    "training_variant"
  ]
  hpo_training_variant = training_variants[hpo_training_variant_name]
  hpo_family = family_for_model(hpo_base_model_name)

  use_all_valid = (
    OPTUNA_VALID_SAMPLE_ROWS is None
    or OPTUNA_VALID_SAMPLE_ROWS >= len(X_valid)
  )
  if use_all_valid:
    X_valid_hpo = X_valid
    y_valid_hpo = y_valid
  else:
    X_valid_hpo = X_valid.sample(
      OPTUNA_VALID_SAMPLE_ROWS,
      random_state=RANDOM_STATE,
    )
    y_valid_hpo = y_valid.loc[X_valid_hpo.index]

  use_all_train = (
    OPTUNA_TRAIN_SAMPLE_ROWS is None
    or OPTUNA_TRAIN_SAMPLE_ROWS >= len(hpo_training_variant["X"])
  )
  if use_all_train:
    X_train_hpo = hpo_training_variant["X"]
    y_train_hpo = hpo_training_variant["y"]
    sample_weight_hpo = hpo_training_variant["sample_weight"]
  else:
    X_train_hpo = hpo_training_variant["X"].sample(
      OPTUNA_TRAIN_SAMPLE_ROWS,
      random_state=RANDOM_STATE,
    )
    y_train_hpo = hpo_training_variant["y"].loc[X_train_hpo.index]
    if hpo_training_variant["sample_weight"] is None:
      sample_weight_hpo = None
    else:
      weight_series = pd.Series(
        hpo_training_variant["sample_weight"],
        index=hpo_training_variant["y"].index,
      )
      sample_weight_hpo = weight_series.loc[
        X_train_hpo.index
      ].to_numpy()

  def hpo_objective(trial):
    if hpo_family == "lightgbm":
      params = suggest_lightgbm_params(trial)
    else:
      params = suggest_xgboost_params(trial)
    model = make_hpo_model(hpo_family, params)
    fit_kwargs = {}
    if sample_weight_hpo is not None:
      fit_kwargs["model__sample_weight"] = sample_weight_hpo
    model.fit(X_train_hpo, y_train_hpo, **fit_kwargs)
    score = get_score(model, X_valid_hpo)
    return average_precision_score(y_valid_hpo, score)

  sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
  optuna.logging.set_verbosity(optuna.logging.WARNING)
  study = optuna.create_study(
    direction="maximize",
    study_name=f"{hpo_base_model_name}_average_precision",
    sampler=sampler,
  )
  study.optimize(
    hpo_objective,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
  )
  best_params_df = pd.Series(study.best_params).to_frame("value")
  hpo_trials_df = study.trials_dataframe().sort_values(
    "value",
    ascending=False,
  )
  best_params_export = best_params_df.reset_index().rename(
    columns={"index": "parameter"},
  )
  save_plot_data(best_params_export, "hyperparameter_best_params")
  save_plot_data(hpo_trials_df, "hyperparameter_trials")

  print("Optimization base model:", hpo_base_model_name)
  print("Model family:", hpo_family)
  print("Training variant:", hpo_training_variant_name)
  print("Best validation average precision:", study.best_value)
  display(best_params_df)
  display(hpo_trials_df.head(10))


In [ ]:
if SKIP_TRAINING_AND_OPTIMIZATION:
  best_model = fitted_models[best_model_name]
  print("Best cached model:", best_model_name)
else:
  optimized_model_name = (
    f"{hpo_base_model_name}_optimized_average_precision"
  )
  optimized_model = make_hpo_model(hpo_family, study.best_params)

  fit_kwargs = {}
  if hpo_training_variant["sample_weight"] is not None:
    fit_kwargs["model__sample_weight"] = hpo_training_variant[
      "sample_weight"
    ]

  optimized_model.fit(
    hpo_training_variant["X"],
    hpo_training_variant["y"],
    **fit_kwargs,
  )
  fitted_models[optimized_model_name] = optimized_model
  optimized_metrics = ranking_metrics(
    optimized_model_name,
    optimized_model,
    X_valid,
    y_valid,
  )
  results_df = pd.concat(
    [results_df, pd.DataFrame([optimized_metrics])],
    ignore_index=True,
  ).sort_values("average_precision", ascending=False)

  best_model_name = results_df.iloc[0]["model"]
  best_model = fitted_models[best_model_name]
  cache = {
    "schema_version": 1,
    "feature_set": "without_meta_features",
    "fitted_models": fitted_models,
    "results_df": results_df,
    "training_timings_df": training_timings_df,
    "optimized_model_name": optimized_model_name,
    "best_model_name": best_model_name,
    "feature_columns": feature_cols,
  }
  joblib.dump(cache, WORKFLOW_CACHE_PATH)
  print(f"saved: {WORKFLOW_CACHE_PATH}")

save_plot_data(results_df, "validation_all_model_metrics")
display(results_df)
print("Best model after average precision optimization:", best_model_name)


## 13. Validation model comparison

The left subplot keeps one row for each model class and highlights the selected class before and after hyperparameter optimization. The right subplot shows validation ROC curves for the class winners.

In [ ]:
from sklearn.metrics import roc_curve


def model_class_label(model_name):
  if model_name.startswith("dummy"):
    return "Dummy prior"
  if model_name.startswith("xgboost_unweighted"):
    return "XGBoost unweighted"
  if model_name.startswith("xgboost_weighted"):
    return "XGBoost weighted"
  if model_name.startswith("lightgbm_unweighted"):
    return "LightGBM unweighted"
  if model_name.startswith("lightgbm_weighted"):
    return "LightGBM weighted"
  return model_name


model_order = [
  "Dummy prior",
  "XGBoost unweighted",
  "XGBoost weighted",
  "LightGBM unweighted",
  "LightGBM weighted",
]

comparison_df = results_df.copy()
comparison_df["model_class"] = comparison_df["model"].map(
  model_class_label
)
comparison_df["is_optimized"] = comparison_df["model"].eq(
  optimized_model_name
)
class_winners = (
  comparison_df
  .sort_values("average_precision", ascending=False)
  .groupby("model_class", as_index=False)
  .head(1)
)
class_winners["model_class"] = pd.Categorical(
  class_winners["model_class"],
  categories=model_order,
  ordered=True,
)
class_winners = class_winners.sort_values("model_class")

optimized_class = model_class_label(optimized_model_name)
before_optimization = (
  comparison_df[
    comparison_df["model_class"].eq(optimized_class)
    & ~comparison_df["model"].eq(optimized_model_name)
    & ~comparison_df["model"].str.startswith("dummy")
  ]
  .sort_values("average_precision", ascending=False)
  .head(1)
)

plot_points = class_winners.copy()
plot_points["point_type"] = np.where(
  plot_points["model"].eq(optimized_model_name),
  "After optimization",
  "Class winner",
)
if not before_optimization.empty:
  before_optimization = before_optimization.copy()
  before_optimization["point_type"] = "Before optimization"
  plot_points = pd.concat(
    [plot_points, before_optimization],
    ignore_index=True,
  )

save_plot_data(class_winners, "validation_comparison_model_rows")
save_plot_data(plot_points, "validation_comparison_plot_points")

display(class_winners[[
  "model_class",
  "model",
  "average_precision",
  "average_precision_lift_over_random",
  "roc_auc",
]])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5), constrained_layout=True)
y_lookup = {label: i for i, label in enumerate(model_order)}
point_offsets = {
  "Class winner": 0.0,
  "Before optimization": -0.18,
  "After optimization": 0.18,
}
label_offsets = {
  "Class winner": 0.0,
  "Before optimization": -0.36,
  "After optimization": 0.36,
}
marker_style = {
  "Class winner": {
    "marker": "o",
    "s": 78,
    "label": "Class winner",
    "edgecolor": "white",
    "linewidth": 0.9,
    "zorder": 3,
  },
  "Before optimization": {
    "marker": "X",
    "s": 70,
    "label": "Best baseline before optimization",
    "edgecolor": "black",
    "linewidth": 1.1,
    "zorder": 5,
  },
  "After optimization": {
    "marker": "D",
    "s": 70,
    "label": "Best model after optimization",
    "edgecolor": "black",
    "linewidth": 1.1,
    "zorder": 6,
  },
}

for point_type, style in marker_style.items():
  subset = plot_points[plot_points["point_type"].eq(point_type)]
  if subset.empty:
    continue
  y_values = (
    subset["model_class"].map(y_lookup).astype(float)
    + point_offsets[point_type]
  )
  axes[0].scatter(
    subset["average_precision_lift_over_random"],
    y_values,
    marker=style["marker"],
    s=style["s"],
    label=style["label"],
    edgecolors=style["edgecolor"],
    linewidths=style["linewidth"],
    zorder=style["zorder"],
  )

axes[0].axvline(
  1.0,
  linestyle="--",
  linewidth=1,
  color="0.25",
  label="Random ranking",
  zorder=1,
)
axes[0].set_yticks(np.arange(len(model_order)))
axes[0].set_yticklabels(model_order)
axes[0].invert_yaxis()
axes[0].set_xlabel("Average precision lift over random")
axes[0].set_title("Validation average precision ranking")
x_values = plot_points["average_precision_lift_over_random"].to_numpy()
axes[0].set_xlim(0.85, max(2.15, x_values.max() + 0.30))
x_min, x_max = axes[0].get_xlim()
x_text_nudge = 0.018 * (x_max - x_min)
for row in plot_points.itertuples(index=False):
  value = row.average_precision_lift_over_random
  y_marker = y_lookup[row.model_class] + point_offsets[row.point_type]
  y_label = y_lookup[row.model_class] + label_offsets[row.point_type]
  x_label = min(value + x_text_nudge, x_max - 0.025 * (x_max - x_min))
  axes[0].annotate(
    f"{value:.3f}x",
    xy=(value, y_marker),
    xytext=(x_label, y_label),
    textcoords="data",
    ha="left",
    va="center",
    fontsize=9,
    bbox={
      "boxstyle": "round,pad=0.18",
      "facecolor": "white",
      "edgecolor": "0.8",
      "linewidth": 0.5,
      "alpha": 0.92,
    },
    arrowprops={
      "arrowstyle": "-",
      "color": "0.45",
      "linewidth": 0.7,
      "shrinkA": 2,
      "shrinkB": 3,
    },
    zorder=7,
  )
axes[0].grid(axis="x", linestyle=":", alpha=0.4)
axes[0].legend(
  loc="lower left",
  bbox_to_anchor=(0.01, 0.01),
  borderaxespad=0,
  fontsize=8,
  frameon=True,
  facecolor="white",
  edgecolor="0.85",
)

roc_rows = []
for row in class_winners.itertuples(index=False):
  score = get_score(fitted_models[row.model], X_valid)
  fpr, tpr, _ = roc_curve(y_valid, score)
  roc_rows.append(pd.DataFrame({
    "model_class": str(row.model_class),
    "model": row.model,
    "false_positive_rate": fpr,
    "true_positive_rate": tpr,
    "roc_auc": row.roc_auc,
  }))
  axes[1].plot(
    fpr,
    tpr,
    linewidth=1.8,
    label=f"{row.model_class}: AUC = {row.roc_auc:.4f}",
    zorder=2,
  )
axes[1].plot(
  [0, 1],
  [0, 1],
  linestyle=":",
  color="black",
  linewidth=2.2,
  label="Random: AUC = 0.5000",
  zorder=8,
)
axes[1].set_xlabel("False-positive rate")
axes[1].set_ylabel("True-positive rate")
axes[1].set_title("Validation ROC curves")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.01)
axes[1].grid(linestyle=":", alpha=0.4)
axes[1].legend(
  loc="lower right",
  fontsize=8,
  frameon=True,
  facecolor="white",
  edgecolor="0.85",
)

fig.suptitle("Validation model comparison", fontsize=14)
save_figure(fig, "validation_model_comparison")
plt.show()

roc_curves_df = pd.concat(roc_rows, ignore_index=True)
save_plot_data(roc_curves_df, "validation_roc_curves")


## 14. Calibration metrics by decision threshold

The production operating threshold is selected on the calibration split, not on validation or test. The selected threshold maximizes calibration-set F1. The maximum-MCC threshold is shown as an operating-point reference, but MCC is not used for model optimization.

In [ ]:
calibration_score = get_score(best_model, X_calibration)
threshold_table = threshold_metric_table(y_calibration, calibration_score)
max_f1_row = best_threshold_row(threshold_table, "f1")
max_mcc_row = best_threshold_row(threshold_table, "mcc")
selected_threshold = float(max_f1_row["threshold"])
threshold_reason = "maximum calibration-set F1"

threshold_summary_df = pd.DataFrame([
  {"threshold_policy": "maximum calibration F1", **max_f1_row.to_dict()},
  {
    "threshold_policy": "maximum calibration MCC",
    **max_mcc_row.to_dict(),
  },
])

save_plot_data(threshold_table, "calibration_threshold_curves")
save_plot_data(threshold_summary_df, "calibration_threshold_summary")

print("Selected model:", best_model_name)
print("Selected threshold:", selected_threshold)
print("Threshold reason:", threshold_reason)
display(threshold_summary_df)

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(
  threshold_table["threshold"],
  threshold_table["precision"],
  label="Precision",
)
ax.plot(
  threshold_table["threshold"],
  threshold_table["recall"],
  label="Recall",
)
ax.plot(threshold_table["threshold"], threshold_table["f1"], label="F1")
ax.plot(threshold_table["threshold"], threshold_table["mcc"], label="MCC")
ax.axvline(
  max_f1_row["threshold"],
  color="black",
  linestyle="-",
  linewidth=1.9,
  label="Maximum calibration F1",
)
ax.axvline(
  max_mcc_row["threshold"],
  color="black",
  linestyle="--",
  linewidth=1.9,
  label="Maximum calibration MCC",
)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Calibration metric")
ax.set_title("Calibration metric by decision threshold")
ax.grid(linestyle=":", alpha=0.4)
ax.legend(
  loc="center left",
  bbox_to_anchor=(1.02, 0.5),
  fontsize=8,
  frameon=True,
  facecolor="white",
  edgecolor="0.85",
)
fig.tight_layout()
save_figure(fig, "calibration_threshold_selection")
plt.show()


## 15. Final test evaluation and confusion matrix

The test split is evaluated once using the selected model and calibration threshold.

In [ ]:
test_score = get_score(best_model, X_test)
test_pred = (test_score >= selected_threshold).astype(int)
test_prevalence = y_test.mean()
test_average_precision = average_precision_score(y_test, test_score)

test_metrics = {
  "model": best_model_name,
  "threshold": selected_threshold,
  "threshold_reason": threshold_reason,
  "prevalence": float(test_prevalence),
  "roc_auc": float(roc_auc_score(y_test, test_score)),
  "average_precision": float(test_average_precision),
  "average_precision_lift_over_random": float(
    test_average_precision / test_prevalence
  ),
  "balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
  "precision": float(precision_score(y_test, test_pred)),
  "recall": float(recall_score(y_test, test_pred)),
  "f1": float(f1_score(y_test, test_pred)),
  "mcc": float(matthews_corrcoef(y_test, test_pred)),
  "predicted_positive_rate": float(test_pred.mean()),
}

test_metrics_df = pd.DataFrame([test_metrics])
save_plot_data(test_metrics_df, "test_metrics")
display(test_metrics_df)
print(classification_report(y_test, test_pred, digits=4))

cm = confusion_matrix(y_test, test_pred)
confusion_matrix_df = pd.DataFrame(
  cm,
  index=["actual_0", "actual_1"],
  columns=["predicted_0", "predicted_1"],
)
confusion_matrix_export = confusion_matrix_df.reset_index().rename(
  columns={"index": "actual_class"}
)
save_plot_data(
  confusion_matrix_export,
  "test_confusion_matrix_counts",
)

fig, ax = plt.subplots(figsize=(5.8, 5.0))
ConfusionMatrixDisplay(cm).plot(ax=ax, values_format=",d", colorbar=False)
ax.set_title(f"Test confusion matrix\n{best_model_name}")
fig.tight_layout()
save_figure(fig, "test_confusion_matrix")
plt.show()


# Additional test-set ranking plots, matching the old production notebook outputs.
test_fpr, test_tpr, _ = roc_curve(y_test, test_score)
test_roc_curve_df = pd.DataFrame({
  "false_positive_rate": test_fpr,
  "true_positive_rate": test_tpr,
  "roc_auc": roc_auc_score(y_test, test_score),
  "model": best_model_name,
})
save_plot_data(test_roc_curve_df, "test_roc_curve")

fig, ax = plt.subplots(figsize=(6.2, 5.2))
RocCurveDisplay.from_predictions(
  y_test,
  test_score,
  ax=ax,
  name=best_model_name,
)
ax.plot(
  [0, 1],
  [0, 1],
  linestyle=":",
  color="black",
  linewidth=2.0,
  label="Random",
)
ax.set_title("Test ROC curve")
ax.grid(linestyle=":", alpha=0.4)
ax.legend(
  loc="lower right",
  fontsize=8,
  frameon=True,
  facecolor="white",
  edgecolor="0.85",
)
fig.tight_layout()
save_figure(fig, "test_roc_curve")
plt.show()

test_precision, test_recall, test_pr_thresholds = (
  precision_recall_curve(y_test, test_score)
)
test_precision_recall_curve_df = pd.DataFrame({
  "recall": test_recall,
  "precision": test_precision,
  "threshold": np.r_[test_pr_thresholds, np.nan],
  "random_baseline": test_prevalence,
  "model": best_model_name,
})
save_plot_data(
  test_precision_recall_curve_df,
  "test_precision_recall_curve",
)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
PrecisionRecallDisplay.from_predictions(
  y_test,
  test_score,
  ax=ax,
  name=best_model_name,
)
ax.axhline(
  test_prevalence,
  linestyle=":",
  color="black",
  linewidth=2.0,
  label=f"Random baseline = {test_prevalence:.3f}",
)
ax.set_title("Test precision-recall curve")
ax.grid(linestyle=":", alpha=0.4)
ax.legend(
  loc="upper right",
  fontsize=8,
  frameon=True,
  facecolor="white",
  edgecolor="0.85",
)
fig.tight_layout()
save_figure(fig, "test_precision_recall_curve")
plt.show()


## 16. Feature importance

Permutation importance is computed on a capped validation sample using average precision as the scoring metric. This measures how much validation ranking quality drops when each feature is shuffled.

In [ ]:
IMPORTANCE_SAMPLE_ROWS = 100_000
IMPORTANCE_REPEATS = 8

importance_n = min(IMPORTANCE_SAMPLE_ROWS, len(X_valid))
X_importance = X_valid.sample(importance_n, random_state=RANDOM_STATE)
y_importance = y_valid.loc[X_importance.index]

perm = permutation_importance(
  best_model,
  X_importance,
  y_importance,
  n_repeats=IMPORTANCE_REPEATS,
  scoring="average_precision",
  random_state=RANDOM_STATE,
  n_jobs=1,
)

importance_df = pd.DataFrame({
  "feature": feature_cols,
  "importance_mean": perm.importances_mean,
  "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

save_plot_data(importance_df, "validation_permutation_importance")
display(importance_df.head(30))

plot_df = importance_df.head(24).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.2, 7.0))
ax.barh(
  plot_df["feature"],
  plot_df["importance_mean"],
  xerr=plot_df["importance_std"],
)
ax.set_xlabel("Decrease in validation average precision")
ax.set_title(f"Permutation feature importance\n{best_model_name}")
fig.tight_layout()
save_figure(fig, "validation_permutation_importance")
plt.show()


## 17. Leakage and split checks

In [ ]:
forbidden_features = {
  "future_blunder_count",
  "will_blunder_soon",
  "result",
  "result_white",
  "final_phase_progress",
  "mean_phase_progress",
  "max_phase_progress",
  "final_total_non_pawn_material",
  "final_material_imbalance_white",
}

used_forbidden = forbidden_features.intersection(feature_cols)
assert not used_forbidden, used_forbidden

split_groups = {
  "train": set(g_train),
  "valid": set(g_valid),
  "calibration": set(g_calibration),
  "test": set(g_test),
}

split_names = list(split_groups)
for i, left in enumerate(split_names):
  for right in split_names[i + 1:]:
    assert split_groups[left].isdisjoint(split_groups[right])

for split_y in [y_valid, y_calibration, y_test]:
  assert split_y.mean() < 0.5

print("No forbidden feature names are used.")
print("Train/validation/calibration/test games are disjoint.")
print("Evaluation splits retain natural class prevalence.")


## 18. Save selected models and metadata

In [ ]:
models_to_save = {}
for row in class_winners.itertuples(index=False):
  if str(row.model).startswith("dummy"):
    continue
  safe_class = (
    str(row.model_class)
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
  )
  models_to_save[safe_class] = row.model
models_to_save["overall_best"] = best_model_name

saved_model_paths = {}
for label, model_name in models_to_save.items():
  path = MODELS_DIR / f"{label}.joblib"
  joblib.dump(fitted_models[model_name], path)
  saved_model_paths[label] = str(path)
  print(f"saved: {path}")

metadata = {
  "created_utc": datetime.now(timezone.utc).isoformat(),
  "target": "will_blunder_soon",
  "horizon_own_moves": HORIZON_OWN_MOVES,
  "blunder_pawn_loss_threshold": BLUNDER_PAWN_LOSS,
  "optimization_metric": "average_precision",
  "best_model_name": best_model_name,
  "selected_threshold": selected_threshold,
  "threshold_reason": threshold_reason,
  "feature_columns": feature_cols,
  "split_summary": split_summary.to_dict(orient="records"),
  "validation_results": results_df.to_dict(orient="records"),
  "test_metrics": test_metrics,
  "saved_models": saved_model_paths,
  "figure_directory": str(FIGURES_DIR),
  "plot_data_directory": str(PLOT_DATA_DIR),
}
metadata_path = MODELS_DIR / "model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"saved: {metadata_path}")


## 19. Plot-data workstation

The figure source data are exported as CSV files under `plot-data/blunder-5-moves`. Use this cell as a lightweight workstation for small plot adjustments without rerunning model training.

In [ ]:
available_plot_data = sorted(PLOT_DATA_DIR.glob("*.csv"))
print("Exported plot-data files:")
for path in available_plot_data:
  print(path.name)

# Example: reload the validation comparison points for quick plot tweaks.
validation_plot_points = pd.read_csv(
  PLOT_DATA_DIR / "validation_comparison_plot_points.csv"
)
display(validation_plot_points.head())
